In [6]:
# fast_block_shuffle_loader.py
import math, os
import numpy as np
import h5py
import torch
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

# ---- HDF5 open helper (single-process read; no SWMR needed) ----
def _open_h5(h5_path: str):
    return h5py.File(
        h5_path, "r",
        libver="latest",
        rdcc_nslots=1_000_003, rdcc_nbytes=128 * 1024 * 1024, rdcc_w0=0.75
    )

class H5BlockShuffleBatches(IterableDataset):
    """
    Fast randomized batches from HDF5:
      - reads contiguous BLOCKS from disk (fast I/O),
      - shuffles the block in RAM (randomization),
      - yields (B,1,32,32), (B,4), (B,1) batches.

    Args:
      h5_path: path to train_test_split.h5
      split  : "train" or "test"
      batch_size: per-yield batch size
      block_size: number of samples per contiguous disk read
      seed: base RNG seed (each worker gets a derived seed)
      cast_patterns_to: np.float32 (default) or np.float16 if you want smaller RAM (then .float() in torch)
      warm_params: (touch_params, touch_neff) lightly touches metadata first block to warm cache

    Notes:
      - Use num_workers=0 on WSL; on Linux you can try 2–4 with persistent_workers=True.
      - Keep file on /home/... (ext4), not /mnt/c/...
    """
    def __init__(
        self,
        h5_path: str,
        split: str = "train",
        batch_size: int = 512,
        block_size: int = 32768,
        seed: int = 1337,
        cast_patterns_to=np.float32,
        warm_params: tuple[bool, bool] = (True, True),
    ):
        assert split in ("train", "test")
        self.h5_path = h5_path
        self.split = split
        self.batch_size = int(batch_size)
        self.block_size = int(block_size)
        self.base_seed = int(seed)
        self.cast_patterns_to = cast_patterns_to
        self.warm_params = warm_params

        with h5py.File(self.h5_path, "r") as f:
            self.N = f[f"pattern_{split}"].shape[0]

    def __len__(self):
        # approximate length (batches), useful for progress bars
        return math.ceil(self.N / self.batch_size)

    def _open(self):
        f = _open_h5(self.h5_path)
        s = self.split
        return f, f[f"pattern_{s}"], f[f"params_{s}"], f[f"neff_{s}"]

    def __iter__(self):
        info = get_worker_info()
        if info is None:
            start, end = 0, self.N
            worker_id, num_workers = 0, 1
        else:
            per_worker = int(math.ceil(self.N / info.num_workers))
            start = info.id * per_worker
            end = min(start + per_worker, self.N)
            worker_id, num_workers = info.id, info.num_workers

        # worker-local RNG
        rng = np.random.default_rng(self.base_seed + worker_id * 7919)

        f, pat, par, neff = self._open()

        # light warmup for metadata to avoid cold stalls
        if any(self.warm_params) and end > start:
            lo = start
            hi = min(start + min(4096, end - start), end)
            if self.warm_params[0]:
                _ = par[lo:hi]
            if self.warm_params[1]:
                _ = neff[lo:hi, 0:1]

        B = self.batch_size
        BS = self.block_size

        for block_lo in range(start, end, BS):
            block_hi = min(block_lo + BS, end)
            bsz = block_hi - block_lo
            if bsz <= 0: break

            # ---- contiguous disk reads (fast) ----
            # patterns (b,32,32)
            p = pat[block_lo:block_hi, :32, :32]
            if p.dtype != self.cast_patterns_to:
                p = p.astype(self.cast_patterns_to, copy=False)
            # params (b,4)
            pr = par[block_lo:block_hi]
            if pr.dtype != np.float32:
                pr = pr.astype(np.float32, copy=False)
            # targets (b,1) : only first neff column
            yv = neff[block_lo:block_hi, 0:1]
            if yv.dtype != np.float32:
                yv = yv.astype(np.float32, copy=False)

            # ---- in-RAM shuffle ----
            perm = rng.permutation(bsz)
            p  = p[perm]
            pr = pr[perm]
            yv = yv[perm]

            # ---- yield mini-batches ----
            # (use float32 tensors on the fly; if cast_patterns_to is float16, cast now)
            for j in range(0, bsz, B):
                jh = min(j + B, bsz)
                x_img = torch.from_numpy(p[j:jh]).unsqueeze(1)
                if x_img.dtype != torch.float32:
                    x_img = x_img.to(torch.float32)
                x_par = torch.from_numpy(pr[j:jh])
                y     = torch.from_numpy(yv[j:jh])
                yield x_img, x_par, y

        f.close()


In [7]:
# paths
H5 = "/home/omiqran/projects/metamaterials_urop/train_test_split.h5"

# dataset & loaders
train_ds = H5BlockShuffleBatches(H5, split="train", batch_size=512, block_size=32768, seed=1337)
val_ds   = H5BlockShuffleBatches(H5, split="test",  batch_size=512, block_size=32768, seed=4242)

# On WSL: keep num_workers=0; on Linux try 2–4 with persistent_workers=True
train_loader = DataLoader(train_ds, batch_size=None, num_workers=0, persistent_workers=False, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=None, num_workers=0, persistent_workers=False, pin_memory=False)


In [8]:
import math
from typing import List, Optional, Callable, Sequence, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F


def _safe_groupnorm_groups(ch: int, preferred: int = 8) -> int:
    """
    Choose a number of groups for GroupNorm that divides 'ch'.
    Tries 'preferred', then decrements until it finds a divisor, falling back to 1.
    """
    g = min(preferred, ch)
    while g > 1 and (ch % g != 0):
        g -= 1
    return g


class ConvBlock(nn.Module):
    def __init__(
        self,
        in_ch: int,
        out_ch: int,
        *,
        act: Optional[Callable[[], nn.Module]] = None,
        dropout_p: float = 0.0,
        use_dropout: bool = False,
        norm_groups_preferred: int = 8,
        kernel_size: int = 3,
        padding: int = 1,
        stride: int = 1,
    ):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size, padding=padding, stride=stride)
        g = _safe_groupnorm_groups(out_ch, norm_groups_preferred)
        self.norm = nn.GroupNorm(g, out_ch)
        self.act  = act() if act is not None else nn.GELU()
        self.drop = nn.Dropout(dropout_p) if use_dropout and dropout_p > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x = self.norm(x)
        x = self.act(x)
        x = self.drop(x)
        return x


class ModularCondCNN(nn.Module):
    """
    A modular CNN that ingests an image (B,1,H,W) and a conditioning vector (B,cond_dim),
    tiles the conditioning to the current spatial size, concatenates it, and proceeds deeper.

    Configure depth/width via lists:
      - pre_concat_blocks:  List[Tuple[out_channels, pool_after(bool)]]
      - post_concat_blocks: List[out_channels]  (no pooling here; add if you want)
      - head_dims:          List[int] for MLP layers before the final scalar output

    Dropout:
      - conv_dropout_p, fc_dropout_p
      - conv_drop_every: apply dropout to every k-th conv block (1 = all, 2 = every other, etc.)

    GroupNorm:
      - norm_groups_preferred: target number of groups (auto-adjusted to divide channels)

    Conditioning:
      - cond_dim: dimension of x_cond (defaults to 4). Tiled to match feature spatial size.
    """

    def __init__(
        self,
        *,
        in_channels: int = 1,
        cond_dim: int = 4,
        pre_concat_blocks: Sequence[Tuple[int, bool]] = ((64, True), (128, True), (256, False)),
        post_concat_blocks: Sequence[int] = (256, 256, 256, 256, 256),
        head_dims: Sequence[int] = (512, 128),
        act_factory: Callable[[], nn.Module] = nn.GELU,
        conv_dropout_p: float = 0.22,
        fc_dropout_p: float = 0.22,
        conv_drop_every: int = 2,  # e.g. 2 = every other (like your original)
        norm_groups_preferred: int = 8,
        kernel_size: int = 3,
        padding: int = 1,
        pool_kernel: int = 2,
        pool_stride: int = 2,
    ):
        super().__init__()

        self.cond_dim = cond_dim
        self.pool = nn.MaxPool2d(pool_kernel, pool_stride)
        self.act_factory = act_factory
        self.conv_dropout_p = conv_dropout_p
        self.fc_dropout_p = fc_dropout_p
        self.conv_drop_every = max(1, conv_drop_every)
        self.norm_groups_preferred = norm_groups_preferred
        self.kernel_size = kernel_size
        self.padding = padding

        # ---- Build pre-concat conv stack (optionally pooling after each block) ----
        convs_pre = []
        in_ch = in_channels
        block_idx = 1
        for out_ch, pool_after in pre_concat_blocks:
            use_dropout = (block_idx % self.conv_drop_every == 0)
            convs_pre.append(
                ConvBlock(
                    in_ch, out_ch,
                    act=act_factory,
                    dropout_p=conv_dropout_p,
                    use_dropout=use_dropout,
                    norm_groups_preferred=norm_groups_preferred,
                    kernel_size=kernel_size,
                    padding=padding,
                )
            )
            if pool_after:
                convs_pre.append(nn.MaxPool2d(pool_kernel, pool_stride))
            in_ch = out_ch
            block_idx += 1
        self.pre = nn.Sequential(*convs_pre)

        # ---- After pre, we will concat tiled conditioning: channels += cond_dim ----
        in_ch_after = in_ch + cond_dim

        # ---- Build post-concat conv stack (no pooling by default) ----
        convs_post = []
        for out_ch in post_concat_blocks:
            use_dropout = (block_idx % self.conv_drop_every == 0)
            convs_post.append(
                ConvBlock(
                    in_ch_after, out_ch,
                    act=act_factory,
                    dropout_p=conv_dropout_p,
                    use_dropout=use_dropout,
                    norm_groups_preferred=norm_groups_preferred,
                    kernel_size=kernel_size,
                    padding=padding,
                )
            )
            in_ch_after = out_ch
            block_idx += 1
        self.post = nn.Sequential(*convs_post)

        # ---- Head: infer flatten size lazily, so we need a small probe in forward ----
        # We'll create the linear layers on first forward pass when we know spatial dims.
        self.head_dims = list(head_dims)
        self.fc_layers: Optional[nn.Sequential] = None
        self.final: Optional[nn.Linear] = None

        self.fc_dropout = nn.Dropout(fc_dropout_p) if fc_dropout_p > 0 else nn.Identity()

# Inside class ModularCondCNN
    def _build_head(self, feat: torch.Tensor):
        b, c, h, w = feat.shape
        flat = c * h * w
        layers = []
        in_dim = flat
        for hd in self.head_dims:
            layers.append(nn.Linear(in_dim, hd))
            layers.append(self.act_factory())
            if isinstance(self.fc_dropout, nn.Dropout):
                layers.append(self.fc_dropout)
            in_dim = hd
        self.fc_layers = nn.Sequential(*layers) if layers else nn.Identity()
        self.final = nn.Linear(in_dim, 1)

        # >>> NEW: ensure correct device/dtype
        dev, dt = feat.device, feat.dtype
        self.fc_layers.to(device=dev, dtype=dt)
        self.final.to(device=dev, dtype=dt)


    def forward(self, x_img: torch.Tensor, x_cond: torch.Tensor) -> torch.Tensor:
        """
        x_img:  [B, 1, H, W]    (H=W=32 in your current setup, but not strictly required)
        x_cond: [B, cond_dim]
        """
        # ---- Pre-concat stack ----
        x = self.pre(x_img)

        # ---- Tile conditioning to current spatial size ----
        b, _, h, w = x.shape
        cond = x_cond.view(b, self.cond_dim, 1, 1).expand(-1, -1, h, w)
        x = torch.cat([x, cond], dim=1)

        # ---- Post-concat stack ----
        x = self.post(x)

        # ---- Head (lazy build) ----
        if self.fc_layers is None or self.final is None:
            self._build_head(x)

        x = torch.flatten(x, 1)
        x = self.fc_layers(x)
        out = self.final(x)
        return out


# ---------------------------
# Example: match your original
# ---------------------------
# def example_cfg_original() -> ModularCondCNN:
#     """
#     This config mirrors your original architecture:
#       Pre:
#         - Conv(1->64) + Pool
#         - Conv(64->128) + Pool
#         - Conv(128->256) (no pool)
#       Concat cond_dim=4 --> channels + 4 (256+4=260)
#       Post:
#         - 5 × Conv(256)
#       Head:
#         - [512, 128] -> 1
#       Dropout:
#         - conv: every other block (like your code), p=0.22
#         - fc: p=0.22
#     """
#     return ModularCondCNN(
#         in_channels=1,
#         cond_dim=4,
#         pre_concat_blocks=((64, True), (128, True), (256, False)),
#         post_concat_blocks=(256, 256, 256, 256, 256),
#         head_dims=(512, 128),
#         act_factory=nn.GELU,
#         conv_dropout_p=0.22,
#         fc_dropout_p=0.22,
#         conv_drop_every=2,
#         norm_groups_preferred=8,
#         kernel_size=3,
#         padding=1,
#         pool_kernel=2,
#         pool_stride=2,
#     )


# # ---------------------------
# # Example: wider & deeper
# # ---------------------------
# def example_cfg_wide_deep() -> ModularCondCNN:
#     """
#     A beefier variant for sweeps. Still ends at ~8×8 if you keep two pools early.
#     """
#     return ModularCondCNN(
#         in_channels=1,
#         cond_dim=4,
#         pre_concat_blocks=((96, True), (192, True), (320, False)),
#         post_concat_blocks=(320, 320, 320, 320, 320, 320),
#         head_dims=(768, 192),
#         conv_dropout_p=0.2,
#         fc_dropout_p=0.3,
#         conv_drop_every=2,
#     )

In [10]:
import torch, torch.nn as nn, torch.optim as optim, time, math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = torch.cuda.is_available()
torch.backends.cudnn.benchmark = True

model = ModularCondCNN(
    in_channels=1, cond_dim=4,
    pre_concat_blocks=[(64, True), (128, True), (256, False)],
    post_concat_blocks=[256, 256, 256, 256, 256],
    head_dims=[512, 128],
    act_factory=torch.nn.GELU,
    conv_dropout_p=0.22, fc_dropout_p=0.22, conv_drop_every=2,
    norm_groups_preferred=8, kernel_size=3, padding=1, pool_kernel=2, pool_stride=2,
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
loss_fn = nn.MSELoss()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

EPOCHS = 5
print("Starting training...\n")
for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    model.train()
    running, seen = 0.0, 0

    # approximate length for progress only
    approx_batches = len(train_loader) if hasattr(train_loader, "__len__") else "?"

    for bi, (x_img, x_cond, y) in enumerate(train_loader, 1):
        x_img, x_cond, y = x_img.to(device), x_cond.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            preds = model(x_img, x_cond).squeeze(-1)
            loss  = loss_fn(preds, y.view_as(preds).float())
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        running += loss.item() * y.size(0); seen += y.size(0)

        if isinstance(approx_batches, int) and (bi % 10 == 0 or bi == approx_batches):
            print(f"  train batch {bi}/{approx_batches}", end="\r")

    train_loss = running / max(seen, 1)

    # validation
    model.eval()
    v_running, v_seen = 0.0, 0
    with torch.no_grad():
        for (x_img, x_cond, y) in val_loader:
            x_img, x_cond, y = x_img.to(device), x_cond.to(device), y.to(device)
            with torch.amp.autocast("cuda", enabled=use_amp):
                preds = model(x_img, x_cond).squeeze(-1)
                l = loss_fn(preds, y.view_as(preds).float())
            v_running += l.item() * y.size(0); v_seen += y.size(0)
    val_loss = v_running / max(v_seen, 1)

    print(f"Epoch {epoch:2d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | time={time.time()-t0:.2f}s")


Starting training...

Epoch  1 | train_loss=1.046030 | val_loss=0.244745 | time=224.68s


KeyboardInterrupt: 

In [11]:
MAX_TRAIN_STEPS = 120      # ~120 * 0.044 s ≈ ~5–6 s of compute
MAX_VAL_STEPS   = 40       # keep validation cheap

for epoch in range(1, EPOCHS+1):
    t0 = time.time()

    # ---- TRAIN ----
    model.train()
    running = seen = 0
    data_t = comp_t = 0.0
    prev = time.time()
    for bi, (x_img, x_cond, y) in enumerate(train_loader, 1):
        t1 = time.time(); data_t += (t1 - prev)
        x_img = x_img.to(device, non_blocking=True)
        x_cond = x_cond.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            preds = model(x_img, x_cond).squeeze(-1)
            loss  = loss_fn(preds, y.view_as(preds).float())
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        torch.cuda.synchronize()
        comp_t += (time.time() - t1); prev = time.time()

        running += loss.item() * y.size(0); seen += y.size(0)
        if bi % 20 == 0:
            print(f"[train {bi}] data={data_t:.2f}s compute={comp_t:.2f}s", end="\r")
        if bi >= MAX_TRAIN_STEPS:
            # Drain any partially-prefetched block quickly:
            break

    train_loss = running / max(seen, 1)

    # ---- VAL ----
    model.eval()
    v_running = v_seen = 0
    with torch.no_grad():
        for vi, (x_img, x_cond, y) in enumerate(val_loader, 1):
            x_img = x_img.to(device, non_blocking=True)
            x_cond = x_cond.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                preds = model(x_img, x_cond).squeeze(-1)
                l = loss_fn(preds, y.view_as(preds).float())
            v_running += l.item() * y.size(0); v_seen += y.size(0)
            if vi >= MAX_VAL_STEPS:
                break
    val_loss = v_running / max(v_seen, 1)

    print(f"\nEpoch {epoch:2d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | time={time.time()-t0:.2f}s")


[train 120] data=6.73s compute=4.57s
Epoch  1 | train_loss=0.467286 | val_loss=0.160391 | time=12.10s
[train 120] data=0.46s compute=4.11s
Epoch  2 | train_loss=0.453942 | val_loss=0.117213 | time=5.26s
[train 120] data=0.45s compute=4.09s
Epoch  3 | train_loss=0.465971 | val_loss=0.144144 | time=5.24s
[train 120] data=0.38s compute=4.05s
Epoch  4 | train_loss=0.450849 | val_loss=0.292568 | time=5.16s
[train 120] data=0.40s compute=4.10s
Epoch  5 | train_loss=0.449939 | val_loss=0.145731 | time=5.21s


In [9]:
data_t = comp_t = 0.0
prev = time.time()
MAX_TRAIN_STEPS = 120          # ~120 steps × 0.045 s ≈ ~5–6 s
MAX_VAL_STEPS   = 40

for bi, batch in enumerate(train_loader, 1):
    t1 = time.time(); data_t += (t1 - prev)        # data wait
    x_img, x_cond, y = (t.to(device) for t in batch)
    optimizer.zero_grad(set_to_none=True)
    torch.cuda.synchronize() if use_amp else None
    t2 = time.time()
    with torch.amp.autocast("cuda", enabled=use_amp):
        preds = model(x_img, x_cond).squeeze(-1)
        loss  = loss_fn(preds, y.view_as(preds).float())
    scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
    torch.cuda.synchronize() if use_amp else None
    comp_t += (time.time() - t2)                   # compute
    prev = time.time()
    if bi % 20 == 0:
        print(f"[b{bi}] data={data_t:.2f}s compute={comp_t:.2f}s")
    if bi >= MAX_TRAIN_STEPS: break


[b20] data=0.52s compute=1.01s
[b40] data=0.53s compute=1.68s
[b60] data=0.53s compute=2.35s
[b80] data=0.94s compute=3.02s
[b100] data=0.94s compute=3.68s
[b120] data=0.95s compute=4.36s


In [14]:
# sweep_modular_small.py
import os, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, IterableDataset
import wandb

# ---------- FAST LOADER (prefetch + block shuffle) ----------
import h5py, threading, queue

def _open_h5(path):
    return h5py.File(path, "r",
        libver="latest",
        rdcc_nslots=1_000_003,
        rdcc_nbytes=256*1024*1024,
        rdcc_w0=0.75
    )

class H5BlockShuffleBatchesPrefetch(IterableDataset):
    def __init__(self, h5_path, split="train",
                 batch_size=512, block_size=32768, seed=1337,
                 cast_patterns_to=np.float16, prefetch_blocks=2,
                 max_batches=None, shuffle_block_order=True):
        super().__init__()
        assert split in ("train","test")
        self.h5_path = h5_path
        self.split = split
        self.batch_size = int(batch_size)
        self.block_size = int(block_size)
        self.base_seed = int(seed)
        self.cast_patterns_to = cast_patterns_to
        self.prefetch_blocks = int(prefetch_blocks)
        self.max_batches = max_batches
        self.shuffle_block_order = shuffle_block_order
        with h5py.File(self.h5_path, "r") as f:
            self.N = f[f"pattern_{split}"].shape[0]

    def __len__(self):
        # approximate (for progress bars only)
        return math.ceil(self.N / self.batch_size)

    def _open(self):
        f = _open_h5(self.h5_path)
        s = self.split
        return f, f[f"pattern_{s}"], f[f"params_{s}"], f[f"neff_{s}"]

    def __iter__(self):
        from torch.utils.data import get_worker_info
        info = get_worker_info()
        if info is None:
            start, end, wid = 0, self.N, 0
        else:
            per = int(math.ceil(self.N / info.num_workers))
            start, end, wid = info.id * per, min((info.id + 1) * per, self.N), info.id

        rng = np.random.default_rng(self.base_seed + 7919 * wid)
        f, pat, par, neff = self._open()

        starts = list(range(start, end, self.block_size))
        if self.shuffle_block_order:
            rng.shuffle(starts)

        q = queue.Queue(maxsize=max(1, self.prefetch_blocks))
        STOP = object()

        def loader():
            try:
                for lo in starts:
                    hi = min(lo + self.block_size, end)
                    bsz = hi - lo
                    p  = pat[lo:hi, :32, :32]
                    pr = par[lo:hi]
                    yv = neff[lo:hi, 0:1]
                    if p.dtype != self.cast_patterns_to:
                        p = p.astype(self.cast_patterns_to, copy=False)
                    if pr.dtype != np.float32:
                        pr = pr.astype(np.float32, copy=False)
                    if yv.dtype != np.float32:
                        yv = yv.astype(np.float32, copy=False)
                    perm = rng.permutation(bsz)
                    p, pr, yv = p[perm], pr[perm], yv[perm]
                    x_img_block = torch.from_numpy(p).unsqueeze(1)
                    if x_img_block.dtype != torch.float32:
                        x_img_block = x_img_block.to(torch.float32)
                    x_par_block = torch.from_numpy(pr)
                    y_block     = torch.from_numpy(yv)
                    q.put((x_img_block, x_par_block, y_block))
            finally:
                q.put(STOP)
                f.close()

        th = threading.Thread(target=loader, daemon=True)
        th.start()

        emitted = 0
        B = self.batch_size
        while True:
            item = q.get()
            if item is STOP:
                break
            x_img_block, x_par_block, y_block = item
            bsz = x_img_block.size(0)
            for j in range(0, bsz, B):
                if self.max_batches is not None and emitted >= self.max_batches:
                    # drain quickly and terminate
                    with q.mutex:
                        q.queue.clear()
                    break
                jh = min(j + B, bsz)
                emitted += 1
                yield (x_img_block[j:jh], x_par_block[j:jh], y_block[j:jh])
            if self.max_batches is not None and emitted >= self.max_batches:
                break

# ---------- SWEEP SETUP ----------
PROJECT = "modular-condcnn-small-sweep"
ENTITY  = None  # or "your_wandb_entity"
H5_PATH = "/home/omiqran/projects/metamaterials_urop/train_test_split.h5"

# Default training budget (quick)
EPOCHS            = 5
MAX_TRAIN_STEPS   = 120
MAX_VAL_STEPS     = 100

def set_seed(seed=1337):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def build_loaders(cfg):
    train_ds = H5BlockShuffleBatchesPrefetch(
        H5_PATH, "train",
        batch_size=cfg.batch_size,
        block_size=cfg.block_size,
        seed=cfg.data_seed,
        cast_patterns_to=np.float16,
        prefetch_blocks=2,
        max_batches=MAX_TRAIN_STEPS,         # hard cap (per epoch) via dataset
        shuffle_block_order=True
    )
    val_ds = H5BlockShuffleBatchesPrefetch(
        H5_PATH, "test",
        batch_size=cfg.batch_size,
        block_size=cfg.block_size,
        seed=cfg.data_seed + 123,
        cast_patterns_to=np.float16,
        prefetch_blocks=1,
        max_batches=MAX_VAL_STEPS,
        shuffle_block_order=True
    )
    # WSL: keep workers=0; prefetch thread overlaps I/O
    train_loader = DataLoader(train_ds, batch_size=None, num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=None, num_workers=0, pin_memory=True)
    return train_loader, val_loader

def build_model(cfg):
    # Pre / Post “width” choices are small for quick runs
    model = ModularCondCNN(
        in_channels=1,
        cond_dim=4,
        pre_concat_blocks=[(cfg.pre_w1, True), (cfg.pre_w2, True), (cfg.pre_w3, False)],
        post_concat_blocks=[cfg.post_w]*cfg.post_depth,
        head_dims=[cfg.head_w1, cfg.head_w2],
        act_factory=torch.nn.GELU if cfg.act == "gelu" else torch.nn.ReLU,
        conv_dropout_p=cfg.conv_dropout,
        fc_dropout_p=cfg.fc_dropout,
        conv_drop_every=cfg.conv_drop_every,
        norm_groups_preferred=cfg.norm_groups,
        kernel_size=cfg.kernel_size,
        padding=cfg.kernel_size // 2,
        pool_kernel=2,
        pool_stride=2,
    )
    return model

def train_one_run():
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    torch.backends.cudnn.benchmark = True

    with wandb.init(project=PROJECT, entity=ENTITY, config=DEFAULTS) as run:
        cfg = wandb.config

        set_seed(cfg.seed)

        # Data
        train_loader, val_loader = build_loaders(cfg)

        # Model
        model = build_model(cfg).to(device)
        if cfg.channels_last:
            model = model.to(memory_format=torch.channels_last)
        n_params = count_params(model)

        # Optim
        opt = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        loss_fn = nn.MSELoss()
        scaler = torch.amp.GradScaler("cuda", enabled=use_cuda and cfg.amp)

        wandb.log({"model/params": n_params})

        best_val = float("inf")
        best_path = os.path.join(os.getcwd(), f"best_{run.name}.pt")

        for epoch in range(1, EPOCHS + 1):
            # ---- Train ----
            model.train()
            train_loss, seen = 0.0, 0
            t0 = time.time()
            for step, (x_img, x_cond, y) in enumerate(train_loader, 1):
                if cfg.channels_last:
                    x_img = x_img.to(device, memory_format=torch.channels_last, non_blocking=True)
                else:
                    x_img = x_img.to(device, non_blocking=True)
                x_cond = x_cond.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)

                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", enabled=use_cuda and cfg.amp):
                    preds = model(x_img, x_cond).squeeze(-1)
                    loss  = loss_fn(preds, y.view_as(preds).float())
                scaler.scale(loss).backward()
                scaler.step(opt); scaler.update()

                train_loss += loss.item() * y.size(0)
                seen += y.size(0)
            train_loss /= max(seen, 1)
            train_time = time.time() - t0

            # ---- Val ----
            model.eval()
            val_loss, vseen = 0.0, 0
            with torch.no_grad():
                for (x_img, x_cond, y) in val_loader:
                    if cfg.channels_last:
                        x_img = x_img.to(device, memory_format=torch.channels_last, non_blocking=True)
                    else:
                        x_img = x_img.to(device, non_blocking=True)
                    x_cond = x_cond.to(device, non_blocking=True)
                    y = y.to(device, non_blocking=True)
                    with torch.amp.autocast("cuda", enabled=use_cuda and cfg.amp):
                        preds = model(x_img, x_cond).squeeze(-1)
                        l = loss_fn(preds, y.view_as(preds).float())
                    val_loss += l.item() * y.size(0)
                    vseen += y.size(0)
            val_loss /= max(vseen, 1)

            wandb.log({
                "train/loss": train_loss,
                "val/loss": val_loss,
                "time/train_epoch_s": train_time,
                "epoch": epoch
            })

            # Save best
            if val_loss < best_val:
                best_val = val_loss
                torch.save({
                    "model": model.state_dict(),
                    "config": dict(cfg),
                    "epoch": epoch,
                    "val_loss": val_loss
                }, best_path)

        # Log artifact
        art = wandb.Artifact(f"model_{wandb.run.id}", type="model")
        art.add_file(best_path)
        wandb.log_artifact(art)

# ---------- SWEEP CONFIG (3 values each) ----------
DEFAULTS = dict(
    # data
    batch_size      = 512,
    block_size      = 32768,
    data_seed       = 1337,
    # training
    lr              = 3e-4,
    weight_decay    = 1e-5,
    amp             = True,
    channels_last   = True,
    seed            = 1337,
    # model (width/depth/dropout)
    pre_w1          = 64,
    pre_w2          = 96,
    pre_w3          = 128,
    post_w          = 192,
    post_depth      = 3,
    head_w1         = 384,
    head_w2         = 128,
    conv_dropout    = 0.10,
    fc_dropout      = 0.10,
    conv_drop_every = 2,
    norm_groups     = 8,
    kernel_size     = 3,
    act             = "gelu",   # or "relu"
)

SWEEP = {
    "method": "grid",
    "metric": {"name": "val/loss", "goal": "minimize"},
    "parameters": {
        # data/system knobs
        "batch_size":   {"values": [384, 512, 640]},
        "block_size":   {"values": [16384, 32768, 65536]},
        # optimizer
        "lr":           {"values": [1e-4, 3e-4, 1e-3]},
        "weight_decay": {"values": [0.0, 1e-5, 1e-4]},
        # AMP / memory fmt
        "amp":          {"values": [True]},   # keep on for speed
        "channels_last":{"values": [False, True]},  # try both
        # model widths/depth
        "pre_w1":       {"values": [48, 64, 80]},
        "pre_w2":       {"values": [64, 96, 128]},
        "pre_w3":       {"values": [96, 128, 160]},
        "post_w":       {"values": [128, 192, 256]},
        "post_depth":   {"values": [2, 3, 4]},
        "head_w1":      {"values": [256, 384, 512]},
        "head_w2":      {"values": [96, 128, 192]},
        # regularization / norms
        "conv_dropout": {"values": [0.05, 0.10, 0.20]},
        "fc_dropout":   {"values": [0.05, 0.10, 0.20]},
        "conv_drop_every":{"values": [0, 2, 3]},
        "norm_groups":  {"values": [4, 8, 16]},
        "kernel_size":  {"values": [3, 5, 7]},
        "act":          {"values": ["relu", "gelu", "gelu"]},  # bias toward gelu a bit
        # seeds
        "seed":         {"values": [123, 1337, 4242]},
        "data_seed":    {"values": [123, 1337, 4242]},
    }
}

import os
os.environ["WANDB_START_METHOD"] = "thread"  # fixes multiprocess WSL timeouts
os.environ["WANDB_HTTP_TIMEOUT"] = "120"
os.environ["WANDB_RETRY_MAX"] = "6"
os.environ["WANDB_RETRY_MIN_SECONDS"] = "2"
os.environ["WANDB_RETRY_MAX_SECONDS"] = "60"

if __name__ == "__main__":
    # Robust to flaky network: enable offline if needed
    if os.environ.get("WANDB_MODE", "").lower() == "offline":
        print("W&B offline mode")
    wandb.login()
    sweep_id = wandb.sweep(SWEEP, project=PROJECT, entity=ENTITY)
    # Run up to N runs per agent (CTRL+C to stop early)
    wandb.agent(sweep_id, function=train_one_run, count=None)


wandb: Network error (ReadTimeout), entering retry loop.


KeyboardInterrupt: 

In [ ]:
import os, wandb

# (Optional) comment this out if you want to test ONLINE mode.
# os.environ["WANDB_MODE"] = "offline"   # force offline test
# os.environ["WANDB_START_METHOD"] = "thread"

wandb.login()  # try to authenticate

# Start a very small test run
with wandb.init(project="wandb-connection-test") as run:
    for step in range(3):
        wandb.log({"step": step, "test_metric": step * 2})
    wandb.alert(title="W&B Test", text="This is a test alert.")
    print("Run URL:", run.url)

print("✅ Finished test run successfully.")


Run URL: https://wandb.ai/sam-dowd/wandb-connection-test/runs/1907y9fr


step,▁▅█
test_metric,▁▅█
step,2
test_metric,4


✅ Finished test run successfully.
